# MSPD -- monthly update

MSPD is published once a month, so the overlap buffer is wider than
Auctions'. Re-fetches from a buffer before the last known record_date
through today for both tables, replaces that window, re-saves.

In [1]:
import os
import pandas as pd
from datetime import timedelta
import sys
sys.path.append(os.path.join(os.path.dirname(os.getcwd()), "Helpers"))
from treasury_fiscaldata_helpers import fetch_mspd

BASE = os.path.dirname(os.getcwd())
RAW_PATH_3 = os.path.join(BASE, "Data Input", "Raw Input", "mspd_table3_raw.csv")
RAW_PATH_5 = os.path.join(BASE, "Data Input", "Raw Input", "mspd_table5_raw.csv")
CLEAN_PATH_3 = os.path.join(BASE, "Data Output", "mspd_table3_clean.csv")
CLEAN_PATH_5 = os.path.join(BASE, "Data Output", "mspd_table5_clean.csv")
BUFFER_DAYS = 45

## Where are we

In [2]:
existing3 = pd.read_csv(CLEAN_PATH_3, parse_dates=["record_date"], low_memory=False)
existing5 = pd.read_csv(CLEAN_PATH_5, parse_dates=["record_date"], low_memory=False)
last_date = max(existing3["record_date"].max(), existing5["record_date"].max())
print(f"Last record_date on file: {last_date.date()}")

Last record_date on file: 2026-08-31


## Re-fetch the buffer window through today

In [3]:
refetch_start = (last_date - timedelta(days=BUFFER_DAYS)).date().isoformat()
print(f"Re-fetching {refetch_start} -> latest available")

fresh3 = fetch_mspd(table="table_3", start=refetch_start)
fresh5 = fetch_mspd(table="table_5", start=refetch_start)
print(f"{len(fresh3):,} table_3 rows, {len(fresh5):,} table_5 rows in the refetched window")

Re-fetching 2026-07-17 -> latest available


2,177 table_3 rows, 819 table_5 rows in the refetched window


## Replace the window and save

In [4]:
existing_raw3 = pd.read_csv(RAW_PATH_3, parse_dates=["record_date"], low_memory=False)
existing_raw5 = pd.read_csv(RAW_PATH_5, parse_dates=["record_date"], low_memory=False)

kept_raw3 = existing_raw3[existing_raw3["record_date"] < pd.Timestamp(refetch_start)]
kept_raw5 = existing_raw5[existing_raw5["record_date"] < pd.Timestamp(refetch_start)]

updated_raw3 = pd.concat([kept_raw3, fresh3], ignore_index=True).drop_duplicates().sort_values("record_date").reset_index(drop=True)
updated_raw5 = pd.concat([kept_raw5, fresh5], ignore_index=True).drop_duplicates().sort_values("record_date").reset_index(drop=True)
updated_raw3.to_csv(RAW_PATH_3, index=False)
updated_raw5.to_csv(RAW_PATH_5, index=False)

updated_clean3 = updated_raw3.dropna(axis=1, how="all").drop_duplicates().sort_values("record_date").reset_index(drop=True)
updated_clean5 = updated_raw5.dropna(axis=1, how="all").drop_duplicates().sort_values("record_date").reset_index(drop=True)
updated_clean3.to_csv(CLEAN_PATH_3, index=False)
updated_clean5.to_csv(CLEAN_PATH_5, index=False)

print(f"table_3: {len(updated_raw3):,} raw rows ({len(updated_raw3) - len(existing_raw3):,} net new), {len(updated_clean3):,} clean rows")
print(f"table_5: {len(updated_raw5):,} raw rows ({len(updated_raw5) - len(existing_raw5):,} net new), {len(updated_clean5):,} clean rows")
print("Saved.")

table_3: 219,321 raw rows (0 net new), 219,321 clean rows
table_5: 86,964 raw rows (0 net new), 86,964 clean rows
Saved.
